# 03 - Feature engineering details

Ce notebook reprend la logique du fichier `feature_engineering.py`, mais de facon plus lisible pour comprendre les variables creees.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(BASE_DIR / 'data' / 'customer_churn.csv')
df.head()

,customer_id,gender,age,country,city,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,...,avg_resolution_time,complaint_type,csat_score,escalations,email_open_rate,marketing_click_rate,nps_score,survey_response,referral_count,churn
0,CUST_00001,Male,68,Bangladesh,London,SME,22,Web,Monthly,26,...,13.354360,Service,4.0,0,0.71,0.40,27,Satisfied,1,0
1,CUST_00002,Female,57,Canada,Sydney,Individual,9,Mobile,Monthly,7,...,25.140088,Billing,2.0,0,0.78,0.33,-19,Neutral,2,1
2,CUST_00003,Male,24,Germany,New York,SME,58,Web,Yearly,19,...,27.572928,Service,3.0,0,0.35,0.49,80,Neutral,1,0
3,CUST_00004,Male,49,Australia,Dhaka,Individual,19,Mobile,Yearly,34,...,26.420822,Technical,5.0,1,0.83,0.15,100,Neutral,0,0
4,CUST_00005,Male,65,Bangladesh,Delhi,Individual,52,Web,Monthly,20,...,26.674579,Technical,4.0,0,0.65,0.44,21,Unsatisfied,1,0


## Traitement des plaintes

Les valeurs manquantes de `complaint_type` signifient simplement qu il n y a pas eu de plainte connue. On les remplace donc par `Aucune Plainte`.

In [2]:
data = df.copy()
data['complaint_type'] = data['complaint_type'].fillna('Aucune Plainte')
data['has_complaint'] = (data['complaint_type'] != 'Aucune Plainte').astype(int)
data[['complaint_type', 'has_complaint']].head()

,complaint_type,has_complaint
0,Service,1
1,Billing,1
2,Service,1
3,Technical,1
4,Technical,1


## Variables metier simples

On cree des indicateurs binaires faciles a expliquer : risque paiement, faible satisfaction, inactivite, contrat mensuel.

In [3]:
data['payment_risk'] = ((data['payment_failures'] > 0) | (data['price_increase_last_3m'] == 'Yes')).astype(int)
data['low_satisfaction'] = ((data['nps_score'] < 0) | (data['csat_score'] <= 2) | (data['survey_response'] == 'Unsatisfied')).astype(int)
data['inactive_customer'] = ((data['monthly_logins'] <= 5) | (data['last_login_days_ago'] >= 20)).astype(int)
data['monthly_contract'] = (data['contract_type'] == 'Monthly').astype(int)
data[['payment_risk', 'low_satisfaction', 'inactive_customer', 'monthly_contract']].head()

,payment_risk,low_satisfaction,inactive_customer,monthly_contract
0,1,0,0,1
1,1,1,0,1
2,1,0,1,0
3,0,0,1,0
4,0,1,0,1


## Ratios et scores

Les ratios permettent de comparer les clients plus justement. Par exemple, un client ancien avec 3 tickets support n est pas dans la meme situation qu un nouveau client avec 3 tickets.

In [4]:
tenure_safe = data['tenure_months'].clip(lower=1)
logins_safe = data['monthly_logins'].clip(lower=1)

data['tickets_per_tenure'] = data['support_tickets'] / tenure_safe
data['revenue_per_month'] = data['total_revenue'] / tenure_safe
data['fee_per_login'] = data['monthly_fee'] / logins_safe
data['support_pressure'] = data['support_tickets'] * data['avg_resolution_time']
data['engagement_score'] = (
    data['monthly_logins'] * 0.30
    + data['weekly_active_days'] * 2.0
    + data['features_used'] * 1.5
    + data['usage_growth_rate'] * 10
    - data['last_login_days_ago'] * 0.40
)
data['satisfaction_score'] = (
    data['csat_score'] * 10
    + data['nps_score'] * 0.20
    - data['escalations'] * 5
    - data['has_complaint'] * 8
)
data[['tickets_per_tenure', 'fee_per_login', 'engagement_score', 'satisfaction_score']].head()

,tickets_per_tenure,fee_per_login,engagement_score,satisfaction_score
0,0.181818,1.153846,27.1,37.4
1,0.111111,4.285714,10.0,8.2
2,0.017241,1.052632,16.8,38.0
3,0.157895,0.882353,15.9,57.0
4,0.000000,2.500000,18.6,36.2


## Verification rapide

On compare les moyennes selon `churn`. Le but est de voir si les variables creees apportent une separation entre clients qui restent et clients qui partent.

In [5]:
new_cols = ['has_complaint','payment_risk','low_satisfaction','inactive_customer','monthly_contract','tickets_per_tenure','fee_per_login','support_pressure','engagement_score','satisfaction_score']
data.groupby('churn')[new_cols].mean().round(3)

,has_complaint,payment_risk,low_satisfaction,inactive_customer,monthly_contract,tickets_per_tenure,fee_per_login,support_pressure,engagement_score,satisfaction_score
churn,,,,,,,,,,
0,0.796,0.502,0.517,0.192,0.496,0.086,3.365,28.987,16.891,31.375
1,0.793,0.587,0.667,0.344,0.502,0.185,6.381,28.894,15.402,26.281
